<a href="https://colab.research.google.com/github/erickcorrea3092/antiMBT_QSAR_model/blob/main/1_Dionisio_similitud.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#ANTECEDENTES

Primero instalamos la libreria RDKit

In [ ]:
from IPython.utils import io
import tqdm.notebook
import os, os.path, sys, random, subprocess
total = 100
with tqdm.notebook.tqdm(total=total) as pbar:
    with io.capture_output() as captured:
      pbar.update(0)
      !pip install molplotly
      !pip install rdkit      #__________________
      import pandas as pd
      import numpy as np
      import seaborn as sn
      #---------------------- RDKit packages
      from rdkit.Chem import AllChem
      from rdkit.Chem import rdMolDescriptors
      from rdkit.Chem.Draw import IPythonConsole
      from rdkit.Chem import Draw
      from rdkit import DataStructs
      from rdkit.ML.Cluster import Butina
      from rdkit.Chem import PandasTools
      from rdkit import Chem
      from rdkit.Chem import rdFMCS
      #---------------------- plotly packages
      import plotly.express as px
      import molplotly
      #from tqdm import tqdm.notebook
      #------------------- hide warning
      import warnings
      warnings.filterwarnings('ignore')
      pbar.update(100)

#Cargamos datos

In [ ]:
with tqdm.notebook.tqdm(total=total) as pbar:
    with io.capture_output() as captured:
      pbar.update(0)
      #Cargamos las datos de una dirección de git hub
      #! gdown --id 1cHM9neEhTOZ82UU9HaZkdGdlwE1d4SJT
      #dataset = pd.read_excel('data.xlsx')
      ! gdown --id 1k-LlMo-zsfZMwwm8KjcUIt-JKE-butYZ
      dataset = pd.read_excel("mordred_descriptors_L9.xlsx")
      dataset.shape #vemos los primeros diez datos
      #! gdown --id 1wZp9pou63ElEYyGGjBeC2pDtscgRgCpj#1Uy
      df_y = dataset['Activity']
      #dataset = dataset.assign(Activity=df_y)
      #dataset['Activity'] = dataset['Activity'].round(2)
      pbar.update(100)



#Funciones

In [ ]:
def find_mcs_multiple(smiles_list):
    mols = [Chem.MolFromSmiles(smiles) for smiles in filtered_df.SMILES]

    # Check for invalid SMILES
    if any(mol is None for mol in mols):
        print("Invalid SMILES string provided in the list.")
        return None, None, 0, False

    mcs = rdFMCS.FindMCS(mols)
    mcs_mol = Chem.MolFromSmarts(mcs.smartsString)

    return mcs_mol, mcs.smartsString, mcs.numAtoms, True

In [ ]:
def draw_multiple_mcs(smiles_list):
    mols = [Chem.MolFromSmiles(smiles) for smiles in filtered_df.SMILES]
    mcs_mol, mcs_smarts, mcs_atom_count, success = find_mcs_multiple(filtered_df.SMILES)
    if success:
        highlight_lists = [mol.GetSubstructMatch(mcs_mol) for mol in mols]

        # Get pIC50 values from the original df using indices from filtered_df
        pic50_values = [filtered_df.loc[idx, 'Activity'] for idx in filtered_df.index]

        mols.append(mcs_mol)  # Add MCS to list for drawing
        highlight_lists.append([]) # No highlight for MCS itself

        # Create legends with pIC50 values
        legends = [f"pIC50: {pic50:.2f}, cluster: {cluster_number} " for pic50 in pic50_values]
        legends.append(f"MCS cluster: {cluster_number}") # Legend for MCS

        img = Draw.MolsToGridImage(mols,
                                   legends=legends,
                                   highlightAtomLists=highlight_lists,
                                   subImgSize=(450,450))
        return img
    else:
        return None

In [ ]:
#regresa el MCS entre dos moléculas
def find_mcs_and_draw(mol1, mol2):
    mcs = rdFMCS.FindMCS([mol1, mol2])
    mcs_mol = Chem.MolFromSmarts(mcs.smartsString)
    if mcs_mol:
        # generacion de la imagen
        img = Draw.MolsToGridImage([mol1, mol2, mcs_mol],
                                    #legends=['m', 'a', 'MCS'],
                                   legends=[#'Cluster ID: ' + str(x) +
                                            ', pIC50: ' + str(y) +
                                            ', index: '+ str(z)
                                            for y, z in zip(#dataset.Cluster_number,
                                                               dataset.Activity,
                                                               dataset.Cmpd)],

                                    highlightAtomLists=[
                                        mol1.GetSubstructMatch(mcs_mol),
                                        mol2.GetSubstructMatch(mcs_mol),
                                        [] # No highlighting for the MCS itself
                                    ])
        return img
    else:
        return None

In [ ]:
#df_y = pd.read_excel("Actividad.xlsx")
#dataset = dataset.assign(Activity=df_y)
#dataset['Activity'] = dataset['Activity'].round(2)
#dataset

In [ ]:
# Incluyamos estructuras 2D  de las moleculas para conocer los datos
# https://github.com/rdkit/rdkit/blob/master/rdkit/Chem/PandasTools.py

from rdkit.Chem import PandasTools
PandasTools.AddMoleculeColumnToFrame(dataset,'SMILES', 'Structure')

#TANIMOTO SIMILARITY

In [ ]:
# Calculate the Tanimoto similarity matrix
with tqdm.notebook.tqdm(total=total) as pbar:
    with io.capture_output() as captured:
      pbar.update(0)
      fps = [AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=2048) for mol in dataset["Structure"]]
      pbar.update(50)
      tanimoto_matrix_ = np.zeros((len(fps), len(fps)))
      for i in range(len(fps)):
        for j in range(i, len(fps)):
          # Calculate only upper triangle, it's symmetric
          tanimoto_matrix_[i, j] = DataStructs.TanimotoSimilarity(fps[i], fps[j])
          tanimoto_matrix_[j, i] = tanimoto_matrix_[i, j] # Fill the lower triangle
      pbar.update(50)

In [ ]:
# Assuming 'tanimoto_matrix' is already calculated as in your original code

fig_ = px.imshow(tanimoto_matrix_,
                labels=dict(x="Molecule Index", y="Molecule Index", color="Tanimoto Similarity"),
                title="Tanimoto Similarity Matrix Heatmap",
                color_continuous_scale='Blues', # You can change the color scale
                width=1000,  # Adjust width for better visualization
                height=800) # Adjust height for beBluestter visualization

fig_.update_xaxes(side="bottom")  # Move x-axis label to the bottom
#fig_.show()

## Análisis de la matriz

In [ ]:
idx=273
#print('cluster ID: '+ str(dataset["Cluster_number"][idx]))
print('pIC50: ' + str(dataset["Activity"][idx]))
m = dataset["Structure"][idx]
m

In [ ]:
idx=1289
#print('cluster ID: '+ str(dataset["Cluster_number"][idx]))
print(f'pIC50: ' + str(dataset["Activity"][idx]))
a = dataset["Structure"][idx]
a

In [ ]:
comparison_image = find_mcs_and_draw(m, a)
if comparison_image:
    comparison_image
else:
    print("MCS not found. Cannot generate comparison image.")
comparison_image

# Agrupación de Taylor-Butina (Taylor-Butina Clustering)

In [ ]:
#función de Culsterificación
def TaylorButina_Cluster(fpts, distThresh = 0.5):
    # generate distance matrix:
    dist_matrix = []
    nfpts = len(fpts)
    for i in tqdm.notebook.tqdm(range(1,nfpts)):

        # generate Tanimoto similary
        Tanimoto_coefficients = DataStructs.BulkTanimotoSimilarity(fpts[i],fpts[:i])
        dist_matrix.extend([1 - j for j in Tanimoto_coefficients])

    # cluster the Tanimoto scores using Butina:
    clusters = Butina.ClusterData(dist_matrix, nfpts, distThresh, isDistData=True)
    cluster_id_list = [0] * nfpts
    for idx,cluster in enumerate(clusters,1):
        for member in cluster:
            cluster_id_list[member] = idx
    return cluster_id_list

In [ ]:
with tqdm.notebook.tqdm(total=total) as pbar:
    with io.capture_output() as captured:
      pbar.update(0)
      all_Mfpts = [AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=4096) for mol in dataset.Structure]
      pbar.update(100)


In [ ]:
cluster_id = TaylorButina_Cluster(all_Mfpts, distThresh=0.5)

In [ ]:
# Number of clusters
len(np.unique(cluster_id))

In [ ]:
#creamos un nuevo dataframe donde agregaremos la información del cluster
dataset_cs = dataset[['Cmpd','SMILES','Structure', 'Activity']] # Selecciona algunas columnas y agrega una columna de número de cluster
dataset_cs['Cluster_number'] = cluster_id #Colocamos la identificación del clúster en el dataframe.
dataset_cs.head() #observar los primeros diez elementos del dataframe

#Cluster

In [ ]:
# prompt: haz una grafica de relación entre activity y clusternumber

import plotly.express as px

fig = px.scatter(dataset_cs, y="Activity", x="Cluster_number",
                 title="Relación entre Actividad y Número de Cluster",
                 labels={"Activity": "Actividad", "Cluster_number": "Número de Cluster"})
app = molplotly.add_molecules(
    fig=fig,
    df=dataset_cs,
    smiles_col="SMILES",
    width=250,
    #caption_cols='G',
    #caption_cols=["y_pred", "y_true", "Molecular Weight"],
    #show_coords=False,
)


app.run(mode="inline", port=870, height=250)
#fig.show()

In [ ]:
#Asigna un número a cada cluster y cuenta las estructuras en cada uno
#dataset_cs.groupby('Cluster_number')['Structure'].count()

In [ ]:
# prompt: haz una grafica de barras de cluste_ number y structure

import plotly.express as px

# Assuming 'dataset_cs' is your DataFrame with 'Cluster_number' and 'Structure' columns
cluster_counts = dataset_cs.groupby('Cluster_number')['Structure'].count().reset_index()

fig = px.bar(cluster_counts, x='Cluster_number', y='Structure',
             labels={'Cluster_number': 'Cluster Number', 'Structure': 'Number of Structures'},
             title='Number of Structures per Cluster')
fig.show()

In [ ]:
# Sort clusters
dataset_cs_sorted = dataset_cs.sort_values(["Cluster_number"])#.head(150)

# from rdkit.Chem import Draw
im = Draw.MolsToGridImage([x for x in dataset_cs_sorted["Structure"]][:125], maxMols=200,
                     legends=[str(x) for x in dataset_cs_sorted.Cluster_number], subImgSize=(350,350), molsPerRow=3)
#im

In [ ]:
# prompt: reinicia los indices da dataset:cs_sorted

# Reset index of dataset_cs_sorted
dataset_cs_sorted = dataset_cs_sorted.reset_index(drop=True)

In [ ]:
# prompt: para los smiles de dataset_cs_sorted calculca los indices de tanimoto de manera matricial

from rdkit import DataStructs
import numpy as np

from IPython.utils import io
import tqdm.notebook
import os, os.path, sys, random, subprocess
total = 100
with tqdm.notebook.tqdm(total=total) as pbar:
    with io.capture_output() as captured:
      pbar.update(0)
      # Calculate fingerprints for the molecules in dataset_cs_sorted
      fps = [AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=2048) for mol in dataset_cs_sorted["Structure"]]
      pbar.update(50)
      # Calculate the Tanimoto similarity matrix
      tanimoto_matrix = np.zeros((len(fps), len(fps)))
      for i in range(len(fps)):
          for j in range(i, len(fps)):  # Calculate only upper triangle, it's symmetric
              tanimoto_matrix[i, j] = DataStructs.TanimotoSimilarity(fps[i], fps[j])
              tanimoto_matrix[j, i] = tanimoto_matrix[i, j] # Fill the lower triangle
      pbar.update(50)

In [ ]:
fig = px.imshow(tanimoto_matrix,
                labels=dict(x="Molecule Index", y="Molecule Index", color="Tanimoto Similarity"),
                title="Tanimoto Similarity Matrix Heatmap 2",
                color_continuous_scale='PuRd', # You can change the color scale
                width=1000,  # Adjust width for better visualization
                height=800) # Adjust height for better visualization

fig.update_xaxes(side="bottom")  # Move x-axis label to the bottom
fig.show()

In [ ]:
idx=1
print('cluster ID: '+ str(dataset_cs_sorted["Cluster_number"][idx]))
print('pIC50: ' + str(dataset_cs_sorted["Activity"][idx]))
m1 = dataset_cs_sorted["Structure"][idx]
m1

In [ ]:
idx=2
print('cluster ID: '+ str(dataset_cs_sorted["Cluster_number"][idx]))
print('pIC50: ' + str(dataset_cs_sorted["Activity"][idx]))
m2 = dataset_cs_sorted["Structure"][idx]
m2

In [ ]:
comparison_image = find_mcs_and_draw(m1, m2)
if comparison_image:
    comparison_image
else:
    print("MCS not found. Cannot generate comparison image.")

In [ ]:
comparison_image

# MCS de cluster

In [ ]:
# prompt: de dataset_cs filtra solo aquello elementos con Cluster_number ==1
cluster_number = 7
# Filter the dataset_cs DataFrame to include only elements where Cluster_number is equal to 1
filtered_df = dataset_cs[dataset_cs['Cluster_number'] == cluster_number]
# Now dataset_cs_cluster_1 contains only the rows where 'Cluster_number' is 1
#filtered_df
comparison_image = draw_multiple_mcs(filtered_df.SMILES)

if comparison_image:
  comparison_image
else:
    print("MCS not found or invalid SMILES provided.")

print('-----------')
print(f'Este es el cluster {cluster_number}')
print('-----------')
comparison_image

In [ ]:
# prompt: guarda en una variable la estructura que se forma del mcs

mcs_mol, mcs_smarts, mcs_atom_count, success = find_mcs_multiple(filtered_df.SMILES)
mcs_structure = mcs_mol
mcs_structure

#todoslos MCS

In [ ]:
# Get unique cluster numbers where the number of structures is >= 2
clusters = dataset_cs.groupby('Cluster_number')['Structure'].count().reset_index()
# Filter clusters DataFrame instead of the list
clusters = clusters[clusters['Structure'] == 1]['Cluster_number'].tolist()
# Get unique cluster numbers where the number of structures is >= 2
clusters_to_process = dataset_cs.groupby('Cluster_number')['Structure'].count().reset_index()
clusters_to_process = clusters_to_process[clusters_to_process['Structure'] >= 2]['Cluster_number'].tolist()

mcs_results = {}  # Store MCS results for each cluster




for cluster_number in clusters_to_process[6:16]: #_______AQUI_____________________________________________________________________________________________________
    filtered_df = dataset_cs[dataset_cs['Cluster_number'] == cluster_number]
    mcs_mol, _, _, success = find_mcs_multiple(filtered_df.SMILES)
    if success:
        mcs_results[cluster_number] = mcs_mol
    else:
        mcs_results[cluster_number] = None
        print(f"Could not find MCS for cluster {cluster_number}")

#_______________________________________________________________________________________________________________________________________________________________

mcs_structure = mcs_results
#mcs_structure
# prompt: con mcs_results has un dataframe donde tengas el objeto mol relacionado con sus cluster_number
mcs_results_df = pd.DataFrame(list(mcs_results.items()), columns=['Cluster_number', 'mol'])
#mcs_results_df
# prompt: a mcs_results_df agrega la columna de cuantas moeculas tenia
mcs_results_df['molecule_count'] = mcs_results_df['Cluster_number'].map(dataset_cs.groupby('Cluster_number')['Structure'].count().to_dict())
#mcs_results_df

In [ ]:
a = Draw.MolsToGridImage([x for x in mcs_results_df["mol"]],#[:125],
                         maxMols=200,
                         legends=['Cluster ID: ' + str(x) + #'\n'
                                   ', Freq: ' + str(y) +'\n'
                                   'pMIC: ''\n'
                                   'min: ' + str(round(z,2)) + #'\n'
                                   ', max: ' + str(round(w,2)) #'\n'
                                    for x, y, z, w in zip(mcs_results_df.Cluster_number,
                                                mcs_results_df.molecule_count,
                                                dataset_cs.groupby('Cluster_number')['Activity'].min(), # Changed to dataset_cs
                                                dataset_cs.groupby('Cluster_number')['Activity'].max())], # Changed to dataset_cs
                          subImgSize=(350,450), molsPerRow=5)
a

In [ ]:
# prompt: de dataset_cs toma los valores donde cluster_nuiber solo aparece una vez

# Filter the dataset_cs DataFrame to include only rows where 'Cluster_number' appears only once.
cluster_counts = dataset_cs['Cluster_number'].value_counts()
single_occurrence_clusters = cluster_counts[cluster_counts == 1].index
dataset_cs_filtered = dataset_cs[dataset_cs['Cluster_number'].isin(single_occurrence_clusters)]
#dataset_cs_filtered
# prompt: ordena  de manera crecienta con respecto a la columna Cluster_nummber a dataset_cs_filtered

# Sort the DataFrame by the 'Cluster_number' column in ascending order
dataset_cs_filtered = dataset_cs_filtered.sort_values('Cluster_number')
#dataset_cs_filtered

In [ ]:
b = Draw.MolsToGridImage([x for x in dataset_cs_filtered["Structure"]][:125], maxMols=200,
                     legends=['Cluster ID: ' + str(x) for x in dataset_cs_filtered.Cluster_number],
                     subImgSize=(350,350), molsPerRow=3)
#b

In [ ]:
#mcs_results_df

In [ ]:
# Create a single list for legends
legends = []

# Add legends for mcs_results_df
legends.extend([
  'Cluster ID: ' + str(x) +
    ', freq: ' + str(y) + '\n' +
    '´pMIC: ''\n' +
    'min : ' + str(round(z,2)) +
    ', max : ' + str(round(w,2))
    for x, y, z, w in zip(mcs_results_df.Cluster_number,
                        mcs_results_df.molecule_count,
                        dataset_cs.groupby('Cluster_number')['Activity'].min(),
                        dataset_cs.groupby('Cluster_number')['Activity'].max())
])

# Add legends for dataset_cs_filtered
legends.extend([
    'Cluster ID: ' + str(x) + ', pMIC: ' + str(round(y,2))
    for x, y in zip(dataset_cs_filtered.Cluster_number, dataset_cs_filtered.Activity)
])

# Now use the flattened 'legends' list in MolsToGridImage
combined_image = Draw.MolsToGridImage(
    [x for x in mcs_results_df["mol"]] + [x for x in dataset_cs_filtered["Structure"]],
    legends=legends,
    maxMols=200,
    subImgSize=(350, 350),
    molsPerRow=4
)
combined_image

# Conclusión

Hemos aprendido a calcular huellas dactilares moleculares (claves MACCS, Avalon, pares de átomos, torsiones topológicas y huellas dactilares de Morgan) usando RDKit para una sola molécula o varias moléculas. También vimos cómo calcular el coeficiente de Tanimoto y la agrupación de Taylor-Butina y el filtrado de moléculas por contenido de segmentos

<div style="background-color:lightgreen; border-style: dotted;border-color: yellow;text-align:center;color:blue;font-size:20px;border-radius: 50px;padding: 20px 20px;"><strong> Tomado originalmente de
Gashaw M. Goshu, Ph.D. in Chemistry </strong></div> https://www.youtube.com/watch?v=-oHqQBUyrQ0 , adaptado por el Dr. Erick Correa para el curso de Desarrollo de Fármacos Asistido por Computadora Laboratorio de Síntesis de Fármacos L-9, F.E.S. Zaragoza UNAM

